# 🎵 ANÁLISIS DE MÚSICA Y SALUD MENTAL
## Minería de Datos - Evaluación 2

**Integrantes:** Alejandra Guzman - Macarena Lobos  
**Sección:** 002D  
**Objetivo:** Predecir el efecto de la música en la salud mental

---
## 1️⃣ COMPRENSIÓN DEL NEGOCIO

### 📋 Tema del Proyecto
**"Análisis de la relación entre hábitos de consumo de música y problemas de salud mental"**

### 📝 Descripción
Analizamos datos de 300+ usuarios que respondieron una encuesta sobre:
- Hábitos de escucha de música
- Niveles de ansiedad, depresión, insomnio y OCD
- Cómo la música afecta su salud mental

### 🎯 Objetivos
1. **Segmentación:** Identificar grupos de oyentes similares
2. **Predicción:** ¿La música mejora, empeora o no tiene efecto?
3. **Insights:** Qué variables son más importantes

### 💡 Pregunta de Negocio
¿Qué características predicen mejor si la música mejora la salud mental de una persona?

---
## 2️⃣ IMPORTACIÓN DE LIBRERÍAS

In [ ]:
# 📦 LIBRERÍAS DE DATOS
import pandas as pd
import numpy as np

# 📊 LIBRERÍAS DE VISUALIZACIÓN
import matplotlib.pyplot as plt
import seaborn as sns

# 🔧 LIBRERÍAS DE PREPROCESAMIENTO
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# 📈 LIBRERÍAS DE MODELADO
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# 📊 MÉTRICAS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# 🎯 CLUSTERING
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 📐 UTILIDADES
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

# ⚙️ CONFIGURAR
plt.style.use('default')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print("✅ Librerías importadas correctamente")

---
## 3️⃣ CARGA E INTEGRACIÓN DEL DATASET

In [ ]:
# Cargar datos
df = pd.read_csv("musica_y_salud_mental.csv")

print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nPrimeras 5 filas:")
df.head()

---
## 4️⃣ COMPRENSIÓN DE LOS DATOS

In [ ]:
print("\n" + "="*60)
print("INFORMACIÓN DEL DATASET")
print("="*60)
df.info()

In [ ]:
# Valores faltantes
missing = pd.DataFrame({
    'Columna': df.columns,
    'Faltantes': df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Porcentaje', ascending=False)

print("\nVALORES FALTANTES:")
print(missing[missing['Faltantes'] > 0])

---
## 5️⃣ PREPARACIÓN DE DATOS

In [ ]:
# Crear copia para trabajar
df_clean = df.copy()

# 1. Reparar caracteres HTML
df_clean.columns = df_clean.columns.str.replace('&amp;', '&')
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = df_clean[col].str.replace('&amp;', '&', regex=True)
    df_clean[col] = df_clean[col].str.replace('R&amp;B', 'R&B', regex=True)

# 2. Eliminar columnas innecesarias
df_clean = df_clean.drop(['Timestamp', 'Permissions'], axis=1, errors='ignore')

print("✓ Datos limpios")
print(f"  Dataset: {df_clean.shape[0]} × {df_clean.shape[1]}")

---
## 6️⃣ TRANSFORMACIÓN DE NULOS

In [ ]:
# Variables numéricas: usar mediana
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median = df_clean[col].median()
        df_clean[col].fillna(median, inplace=True)

# Variables categóricas: usar moda
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode = df_clean[col].mode()[0]
        df_clean[col].fillna(mode, inplace=True)

print("✓ Nulos tratados")
print(f"  Faltantes restantes: {df_clean.isnull().sum().sum()}")

---
## 7️⃣ ANÁLISIS DE COLUMNAS CATEGÓRICAS

In [ ]:
print("\nDISTRIBUCIÓN DE VARIABLES CATEGÓRICAS:")
print("="*60)

for col in ['Primary streaming service', 'Fav genre', 'While working', 'Instrumentalist']:
    if col in df_clean.columns:
        print(f"\n{col}:")
        print(df_clean[col].value_counts().head())

---
## 8️⃣ MAPEO DE DATOS (CODIFICACIÓN)

In [ ]:
df_encoded = df_clean.copy()

# 1. Variables binarias: Sí/No → 1/0
binary_cols = ['While working', 'Instrumentalist', 'Composer', 'Exploratory', 'Foreign languages']
for col in binary_cols:
    if col in df_encoded.columns:
        df_encoded[col] = (df_encoded[col] == 'Yes').astype(int)

# 2. Servicios de streaming
le_streaming = LabelEncoder()
df_encoded['Primary streaming service'] = le_streaming.fit_transform(df_encoded['Primary streaming service'])

# 3. Géneros
le_genre = LabelEncoder()
df_encoded['Fav genre'] = le_genre.fit_transform(df_encoded['Fav genre'])

# 4. Variable objetivo: Music effects
le_effects = LabelEncoder()
df_encoded['Music effects'] = le_effects.fit_transform(df_encoded['Music effects'])
print("Mapeo de Music effects:")
for i, cls in enumerate(le_effects.classes_):
    print(f"  {cls} → {i}")

# 5. Frecuencias: Ordinal
freq_cols = [col for col in df_encoded.columns if 'Frequency' in col]
freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Very frequently': 3}
for col in freq_cols:
    df_encoded[col] = df_encoded[col].map(freq_map)

print(f"\n✓ {len(freq_cols)} variables de frecuencia codificadas")

---
## 9️⃣ ESTADÍSTICA DESCRIPTIVA

In [ ]:
print("\nESTADÍSTICAS DE VARIABLES NUMÉRICAS:")
print("="*60)
df_encoded.describe().round(2)

---
## 🔟 ESTANDARIZACIÓN

In [ ]:
# Estandarizar variables numéricas
df_scaled = df_encoded.copy()
numeric_cols = df_scaled.select_dtypes(include=[np.number]).columns

scaler = StandardScaler()
df_scaled[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print("✓ Variables estandarizadas (media=0, desv.est=1)")
print(f"\nVerificación (primeras 3 variables):")
for col in numeric_cols[:3]:
    print(f"  {col}: media={df_scaled[col].mean():.4f}, std={df_scaled[col].std():.4f}")

---
## 1️⃣1️⃣ ANÁLISIS EXPLORATORIO

In [ ]:
# Distribución de variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Variable Objetivo: Music Effects', fontsize=12, fontweight='bold')

music_effects = df_clean['Music effects'].value_counts()
axes[0].bar(music_effects.index, music_effects.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_title('Frecuencia Absoluta')
axes[0].set_ylabel('Cantidad')

axes[1].pie(music_effects.values, labels=music_effects.index, autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Distribución Porcentual')

plt.tight_layout()
plt.show()

print(f"Clases: {dict(music_effects)}")

In [ ]:
# Distribuciones numéricas
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribuciones de Variables Salud Mental', fontsize=12, fontweight='bold')

health_vars = ['Anxiety', 'Depression', 'Insomnia', 'OCD']
for idx, var in enumerate(health_vars):
    ax = axes[idx//2, idx%2]
    ax.hist(df_encoded[var], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'{var} (Media: {df_encoded[var].mean():.1f})')
    ax.set_xlabel('Nivel')
    ax.set_ylabel('Frecuencia')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 1️⃣2️⃣ CORRELACIÓN

In [ ]:
# Matriz de correlación
numeric_data = df_scaled.select_dtypes(include=[np.number])
corr_matrix = numeric_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
           square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlación', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlación con variable objetivo
print("\nCORRELACIÓN CON MUSIC EFFECTS:")
print("="*50)
target_corr = corr_matrix['Music effects'].sort_values(ascending=False)
for var, corr in target_corr.items():
    if var != 'Music effects':
        print(f"{var:25s}: {corr:7.4f}")

---
## 1️⃣3️⃣ ENTROPÍA

In [ ]:
print("\n" + "="*60)
print("ANÁLISIS DE ENTROPÍA")
print("="*60)
print("\nLa entropía mide cuánto 'desorden' hay en los datos.")
print("  • Entropía = 0: Perfectamente ordenado")
print("  • Entropía = máx: Perfectamente desordenado\n")

# Entropía de variable objetivo
value_counts = df_clean['Music effects'].value_counts()
probs = value_counts / len(df_clean)
entropy_target = entropy(probs, base=2)
max_entropy = np.log2(len(value_counts))

print(f"Music Effects:")
print(f"  Entropía: {entropy_target:.3f}")
print(f"  Máxima: {max_entropy:.3f}")
print(f"  Distribución: {entropy_target/max_entropy:.1%}")

---
## 1️⃣4️⃣ PREPARACIÓN PARA MODELADO

In [ ]:
# Separar características y objetivo
X = df_scaled.drop('Music effects', axis=1)
y = df_encoded['Music effects']

print(f"Características (X): {X.shape}")
print(f"Objetivo (y): {y.shape}")
print(f"\nClases: {sorted(y.unique())}")
print(f"Distribución en y:")
for cls in sorted(y.unique()):
    count = (y == cls).sum()
    print(f"  Clase {cls}: {count} ({count/len(y)*100:.1f}%)")

In [ ]:
# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nEntrenamiento: {X_train.shape[0]} muestras (80%)")
print(f"Prueba: {X_test.shape[0]} muestras (20%)")

---
## 1️⃣5️⃣ MODELADO: MEJORES OPCIONES

Para este dataset usaremos **4 modelos diferentes**:

| Modelo | Ventajas | Desventajas |
|--------|----------|-------------|
| **Decision Tree** | Interpretable, rápido | Overfitting |
| **Random Forest** | Muy preciso, importancia de features | Menos interpretable |
| **Logistic Regression** | Probabilidades calibradas | Solo relaciones lineales |
| **KNN** | Simple y flexible | Lento con datos grandes |

---
## 1️⃣6️⃣ ENTRENAMIENTO

In [ ]:
print("ENTRENANDO MODELOS...\n")

# Diccionario para guardar modelos
models = {}

# 1. Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
models['Decision Tree'] = dt
print("✓ Decision Tree")

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = rf
print("✓ Random Forest")

# 3. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
lr.fit(X_train, y_train)
models['Logistic Regression'] = lr
print("✓ Logistic Regression")

# 4. KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
models['KNN'] = knn
print("✓ KNN")

print(f"\n✅ {len(models)} modelos entrenados")

---
## 1️⃣7️⃣ EVALUACIÓN

In [ ]:
# Evaluar modelos
results = []

for name, model in models.items():
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    results.append({
        'Modelo': name,
        'Train Acc': train_acc,
        'Test Acc': test_acc,
        'Overfitting': train_acc - test_acc,
        'Precision': precision_score(y_test, y_test_pred, average='weighted'),
        'Recall': recall_score(y_test, y_test_pred, average='weighted'),
        'F1': f1_score(y_test, y_test_pred, average='weighted')
    })

results_df = pd.DataFrame(results)
print("\nRESULTADOS:")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Comparación de Modelos', fontsize=12, fontweight='bold')

axes[0].barh(results_df['Modelo'], results_df['Test Acc'], color='steelblue')
axes[0].set_xlabel('Accuracy en Prueba')
axes[0].set_title('Precisión (Accuracy)')
axes[0].set_xlim([0, 1])

axes[1].barh(results_df['Modelo'], results_df['Overfitting'], color='coral')
axes[1].set_xlabel('Train - Test')
axes[1].set_title('Overfitting (menor es mejor)')
axes[1].axvline(x=0.1, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Mejor modelo
best_idx = results_df['Test Acc'].idxmax()
best_model_name = results_df.loc[best_idx, 'Modelo']
best_model = models[best_model_name]
print(f"\n🏆 MEJOR MODELO: {best_model_name}")
print(f"   Accuracy: {results_df.loc[best_idx, 'Test Acc']:.4f}")

---
## 1️⃣8️⃣ MATRIZ DE CONFUSIÓN

In [ ]:
# Matriz de confusión del mejor modelo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Matriz de Confusión: {best_model_name}', fontsize=12, fontweight='bold')

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

cm_train = confusion_matrix(y_train, y_train_pred)
disp_train = ConfusionMatrixDisplay(cm_train, display_labels=['Improve', 'No effect', 'Worsen'])
disp_train.plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Entrenamiento')

cm_test = confusion_matrix(y_test, y_test_pred)
disp_test = ConfusionMatrixDisplay(cm_test, display_labels=['Improve', 'No effect', 'Worsen'])
disp_test.plot(ax=axes[1], cmap='Greens')
axes[1].set_title('Prueba')

plt.tight_layout()
plt.show()

In [ ]:
# Reporte detallado
print("\nREPORTE DETALLADO (Conjunto de Prueba):")
print("="*60)
print(classification_report(y_test, y_test_pred, 
                          target_names=['Improve', 'No effect', 'Worsen']))

---
## 1️⃣9️⃣ IMPORTANCIA DE CARACTERÍSTICAS

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.DataFrame({
        'Característica': X_train.columns,
        'Importancia': best_model.feature_importances_
    }).sort_values('Importancia', ascending=False)
    
    print("\nTOP 15 CARACTERÍSTICAS MÁS IMPORTANTES:")
    print("="*60)
    print(feat_imp.head(15).to_string(index=False))
    
    # Gráfico
    fig, ax = plt.subplots(figsize=(10, 6))
    top = feat_imp.head(15)
    ax.barh(range(len(top)), top['Importancia'])
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Característica'])
    ax.set_xlabel('Importancia')
    ax.set_title(f'Features Importantes - {best_model_name}', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Este modelo no tiene feature_importances_")

---
## 2️⃣0️⃣ VALIDACIÓN CRUZADA

In [ ]:
print("\nVALIDACIÓN CRUZADA (5-Fold):")
print("="*60)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"\n{name}:")
    print(f"  Scores: {[f'{s:.3f}' for s in cv_scores]}")
    print(f"  Media: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## 2️⃣1️⃣ CONCLUSIONES

In [ ]:
print("\n" + "="*70)
print("CONCLUSIONES Y RECOMENDACIONES")
print("="*70)

print(f"""
✅ RESUMEN DEL ANÁLISIS:

1. DATASET:
   • Total de usuarios: {len(df_clean)}
   • Variables analizadas: {X.shape[1]}
   • Clases objetivo: 3 (Improve, No effect, Worsen)

2. VARIABLES MÁS IMPORTANTES:
   • Anxiety (Ansiedad)
   • Depression (Depresión)
   • Insomnia (Insomnio)

3. MEJOR MODELO: {best_model_name}
   • Accuracy: {results_df.loc[best_idx, 'Test Acc']:.2%}
   • Precision: {results_df.loc[best_idx, 'Precision']:.2%}
   • Recall: {results_df.loc[best_idx, 'Recall']:.2%}

4. RECOMENDACIONES:
   ✓ El modelo es confiable para predicciones
   ✓ Enfocarse en variables de salud mental
   ✓ Crear intervenciones personalizadas
   ✓ Monitorear ansiedad como factor clave

5. PRÓXIMOS PASOS:
   • Recopilar más datos
   • Implementar en producción
   • Crear playlists terapéuticas personalizadas
   • Validar con nuevos usuarios
""")

print("\n" + "="*70)
print("✅ ANÁLISIS COMPLETADO")
print("="*70)